# XGBoost — Holistic Model (excludes G1, G2)

This notebook trains an XGBoost model using **only** behavioral, demographic, and lifestyle features.
G1 and G2 are excluded to simulate predicting student performance **before any exams are taken**.

This is useful for **early intervention** — identifying at-risk students at the start of the year.

In [1]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import pickle

In [2]:
df = pd.read_csv("../data/student-mat.csv", sep=";")

print(df.head())
print(df.columns)

  school sex  age address famsize Pstatus  Medu  Fedu     Mjob      Fjob  ...  \
0     GP   F   18       U     GT3       A     4     4  at_home   teacher  ...   
1     GP   F   17       U     GT3       T     1     1  at_home     other  ...   
2     GP   F   15       U     LE3       T     1     1  at_home     other  ...   
3     GP   F   15       U     GT3       T     4     2   health  services  ...   
4     GP   F   16       U     GT3       T     3     3    other     other  ...   

  famrel freetime  goout  Dalc  Walc health absences  G1  G2  G3  
0      4        3      4     1     1      3        6   5   6   6  
1      5        3      3     1     1      3        4   5   5   6  
2      4        3      2     2     3      3       10   7   8  10  
3      3        2      2     1     1      5        2  15  14  15  
4      4        3      2     1     2      5        4   6  10  10  

[5 rows x 33 columns]
Index(['school', 'sex', 'age', 'address', 'famsize', 'Pstatus', 'Medu', 'Fedu',
       '

In [3]:
# Encode categorical columns using LabelEncoder
le_dict = {}

for col in df.select_dtypes(include="object").columns:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    le_dict[col] = le

In [4]:
# Avoid data leakage — drop G1, G2, G3
X = df.drop(["G1", "G2", "G3"], axis=1)
y = df["G3"]

print(f"Features: {X.shape[1]}")
print(f"Feature list: {X.columns.tolist()}")

Features: 30
Feature list: ['school', 'sex', 'age', 'address', 'famsize', 'Pstatus', 'Medu', 'Fedu', 'Mjob', 'Fjob', 'reason', 'guardian', 'traveltime', 'studytime', 'failures', 'schoolsup', 'famsup', 'paid', 'activities', 'nursery', 'higher', 'internet', 'romantic', 'famrel', 'freetime', 'goout', 'Dalc', 'Walc', 'health', 'absences']


In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [6]:
model = XGBRegressor(n_estimators=100, learning_rate=0.1)

model.fit(X_train, y_train)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             gamma=None, grow_policy=None, importance_type=None,
             interaction_constraints=None, learning_rate=0.1, max_bin=None,
             max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=None, max_leaves=None,
             min_child_weight=None, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimators=100, n_jobs=None,
             num_parallel_tree=None, random_state=None, ...)

In [7]:
y_pred = model.predict(X_test)

print("MAE:", mean_absolute_error(y_test, y_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))
print("R2:", r2_score(y_test, y_pred))

MAE: 3.039930582046509
RMSE: 3.8035793061305956
R2: 0.29445594549179077


In [8]:
# Save model and encoders
pickle.dump(model, open("../xgb_model.pkl", "wb"))
pickle.dump(le_dict, open("../encoders.pkl", "wb"))

feature_columns = X.columns.tolist()
pickle.dump(feature_columns, open("../features.pkl", "wb"))

print("Model saved successfully!")
print(f"Features saved: {feature_columns}")

Model saved successfully!
Features saved: ['school', 'sex', 'age', 'address', 'famsize', 'Pstatus', 'Medu', 'Fedu', 'Mjob', 'Fjob', 'reason', 'guardian', 'traveltime', 'studytime', 'failures', 'schoolsup', 'famsup', 'paid', 'activities', 'nursery', 'higher', 'internet', 'romantic', 'famrel', 'freetime', 'goout', 'Dalc', 'Walc', 'health', 'absences']


## Prediction Demo

Test the holistic model with sample student inputs — **no G1/G2 grades needed**.

In [9]:
def predict_student_holistic(student_data):
    """Predict G3 using the holistic model (excludes G1, G2)."""
    
    input_df = pd.DataFrame([student_data])
    
    # Encode categoricals using the saved encoders
    for col in le_dict:
        if col in input_df.columns:
            input_df[col] = le_dict[col].transform(input_df[col])
    
    # Align with training features
    input_df = input_df.reindex(columns=X.columns, fill_value=0)
    
    prediction = model.predict(input_df)
    return round(prediction[0], 2)

In [10]:
# -------- Student 1: Motivated, good environment --------
student_1 = {
    "school": "GP", "sex": "M", "age": 18, "address": "U",
    "famsize": "GT3", "Pstatus": "T", "Medu": 4, "Fedu": 4,
    "Mjob": "teacher", "Fjob": "services", "reason": "course",
    "guardian": "mother", "traveltime": 1, "studytime": 3,
    "failures": 0, "schoolsup": "no", "famsup": "yes",
    "paid": "no", "activities": "yes", "nursery": "yes",
    "higher": "yes", "internet": "yes", "romantic": "no",
    "famrel": 4, "freetime": 3, "goout": 2, "Dalc": 1,
    "Walc": 1, "health": 4, "absences": 2
}

pred = predict_student_holistic(student_1)
print(f"Student 1 (Motivated, good environment) → Predicted G3: {pred}")

Student 1 (Motivated, good environment) → Predicted G3: 11.739999771118164


In [11]:
# -------- Student 2: Average student --------
student_2 = {
    "school": "GP", "sex": "F", "age": 17, "address": "U",
    "famsize": "GT3", "Pstatus": "T", "Medu": 3, "Fedu": 3,
    "Mjob": "services", "Fjob": "other", "reason": "home",
    "guardian": "mother", "traveltime": 2, "studytime": 2,
    "failures": 0, "schoolsup": "no", "famsup": "yes",
    "paid": "no", "activities": "no", "nursery": "yes",
    "higher": "yes", "internet": "yes", "romantic": "no",
    "famrel": 4, "freetime": 3, "goout": 3, "Dalc": 1,
    "Walc": 2, "health": 3, "absences": 6
}

pred = predict_student_holistic(student_2)
print(f"Student 2 (Average) → Predicted G3: {pred}")

Student 2 (Average) → Predicted G3: 10.640000343322754


In [12]:
# -------- Student 3: At-risk student --------
student_3 = {
    "school": "MS", "sex": "M", "age": 19, "address": "R",
    "famsize": "GT3", "Pstatus": "A", "Medu": 1, "Fedu": 1,
    "Mjob": "other", "Fjob": "other", "reason": "other",
    "guardian": "other", "traveltime": 3, "studytime": 1,
    "failures": 2, "schoolsup": "yes", "famsup": "no",
    "paid": "no", "activities": "no", "nursery": "no",
    "higher": "no", "internet": "no", "romantic": "yes",
    "famrel": 2, "freetime": 5, "goout": 5, "Dalc": 3,
    "Walc": 4, "health": 2, "absences": 20
}

pred = predict_student_holistic(student_3)
print(f"Student 3 (At-risk) → Predicted G3: {pred}")

Student 3 (At-risk) → Predicted G3: 7.840000152587891


In [13]:
# -------- Student 4: Hardworking despite tough background --------
student_4 = {
    "school": "GP", "sex": "F", "age": 16, "address": "U",
    "famsize": "LE3", "Pstatus": "T", "Medu": 3, "Fedu": 2,
    "Mjob": "health", "Fjob": "services", "reason": "reputation",
    "guardian": "mother", "traveltime": 1, "studytime": 4,
    "failures": 1, "schoolsup": "yes", "famsup": "yes",
    "paid": "yes", "activities": "yes", "nursery": "yes",
    "higher": "yes", "internet": "yes", "romantic": "no",
    "famrel": 5, "freetime": 2, "goout": 2, "Dalc": 1,
    "Walc": 1, "health": 5, "absences": 0
}

pred = predict_student_holistic(student_4)
print(f"Student 4 (Hardworking) → Predicted G3: {pred}")

Student 4 (Hardworking) → Predicted G3: 10.539999961853027
